# Funciones, sus gradientes, espacio de exploración y mínimos globales

Función de Rosenbrock:

$$ f(x,y) = (1-x)^2 + 100(y-x^2)^2 $$

Gradiente:

$$ \nabla f = [2(1-x) - 400x(y-x^2)]\hat{i} + 200(y-x^2)\hat{j} $$

Tiene mínimo global $0$ en $(1,1)$.

Función _Three-Hump Camel_:

$$ f(x,y) = 2x^2 - 1.05x^4 + \tfrac{x^6}{6} + xy + y^2 $$

Gradiente: 

$$ \nabla f = [4x - 4.20x^3 + x^5 + y]\hat{i} + (1 + 2y)\hat{j} $$

Mínimo global $0$ en $(0, 0)$, función definida en $x_i \in [-5, 5]$.

Función _Booth_:

$$ f(x,y) = (x+2y-7)^2 + (2x + y - 5)^2 $$

Gradiente: 

$$ \nabla f = [2(x + 2y - 7) + 4(2x + y - 5)]\hat{i} + [4(x + 2y - 7) + 2(2x + y - 5)]\hat{j} $$

$$ \nabla f = [10x + 8y - 34]\hat{i} + [8x + 10y - 38]\hat{j} $$

Mínimo global $0$ en $(1,3)$, función suele ser evaluada en $x_i \in [-10, 10]$.

Función Styblinski-Tang:

$$ f(x) = \tfrac{1}{2} \sum_{i=1}^d x_i^4 - 16x_i^2 + 5x_i $$

$$ \nabla f = \tfrac{1}{2} \sum_{i=1}^d 4x_i^3 - 32x_i + 5 $$

Yo consideré $ d=2 $ por simplicidad.

Mínimo global $-39.16599d$ en $(-2.903534, \dots, -2.903534)$.

Suele ser evaluada en el hípercubo $x_i \in [-5, 5]$

Vamos a explorar todas en el segmento $x,y \in [-5, 5]$

Importamos las bibliotecas de siempre

In [154]:
import torch
import sklearn
import matplotlib.pyplot as plt
import numpy as np 

In [155]:
DIMS = 2 # Vamos a usar 2 dimensiones para todas las funciones
torch.manual_seed(21) # Para resultados reproducibles

# Codificando las funciones (criterios (pérdida))

In [156]:
def rosenbrock(x, y):
    return (1-x)**2 + 100 * (y - x**2)**2

def three_hump_camel(x, y):
    return 2*x**2 - 1.05 * x**4 + (1 / 6)*x**6 + x*y + y**2

def booth(x, y):
    return (x + 2*y - 7)**2 + (2*x + y - 5)**2

def styblinski_tang(*args):
    xis = torch.stack(list(args))

    return 0.5*(xis**4 - 16*xis**2 + 5*xis).sum()

Ya que queremos comparar los diferentes métodos de descenso haremos que sea más o menos justo y le daremos el mismo punto de pártida a todos los métodos:

In [157]:
puntos_iniciales = np.random.normal(size=(DIMS,))
puntos_iniciales

array([-1.06174058,  1.131436  ])

In [158]:
puntos_iniciales2 = np.random.normal(size=(DIMS,))
puntos_iniciales2

array([-1.41864486, -0.0642408 ])

# Descenso de gradiente

In [159]:
def gd(punto_partida, criterio, lr=1e-3, n_epocas=100):
    print(f"Punto de partida: {punto_partida[0]}, {punto_partida[1]}")
    punto_actual = punto_partida.clone().detach().requires_grad_(True)
    historial = [punto_actual.detach()]
    for epoca in range(n_epocas):

        if punto_actual.grad is not None:
            punto_actual.grad.zero_()

        perdida = criterio(*punto_actual)
        perdida.backward()
        #optimizador.paso() no podemos utilizar el paso :c 
        # lo que vamos a hacer es hacer el descenso de gradiente manual con los valores de backward
        with torch.no_grad():
            punto_actual -= punto_actual.grad * lr
        

        historial.append(punto_actual.clone().detach())

        if epoca % 5 == 0 or epoca < 5:
            print(f"Pérdida {perdida.item()} en época {epoca + 1} en el punto ({punto_actual[0]}, {punto_actual[1]})")

    return punto_actual, historial

Ahora probaremos las funciones con lr, que parece funcionar más o menos bien 

## Descenso de gradiente en __Rosenbrock__

In [160]:
optimo, historial = gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), rosenbrock, 1e-3)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 4.252490520477295 en época 1 en el punto (-1.05937659740448, 1.1306074857711792)
Pérdida 4.247969150543213 en época 2 en el punto (-1.0587871074676514, 1.1289417743682861)
Pérdida 4.244863986968994 en época 3 en el punto (-1.0580202341079712, 1.127359390258789)
Pérdida 4.241771221160889 en época 4 en el punto (-1.057269811630249, 1.1257688999176025)
Pérdida 4.238678455352783 en época 5 en el punto (-1.0565171241760254, 1.1241790056228638)
Pérdida 4.235583782196045 en época 6 en el punto (-1.055764079093933, 1.122588872909546)
Pérdida 4.220090389251709 en época 11 en el punto (-1.0519894361495972, 1.1146363019943237)
Pérdida 4.204564094543457 en época 16 en el punto (-1.0481996536254883, 1.1066803932189941)
Pérdida 4.1890034675598145 en época 21 en el punto (-1.0443944931030273, 1.0987211465835571)
Pérdida 4.173408031463623 en época 26 en el punto (-1.0405738353729248, 1.0907585620880127)
Pérdida 4.157778739929199 en époc

In [161]:
optimo, historial = gd(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), rosenbrock, 1e-3)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 437.1572570800781 en época 1 en el punto (-0.23531413078308105, 0.3511180281639099)
Pérdida 10.272527694702148 en época 2 en el punto (-0.2606807351112366, 0.29196897149086)
Pérdida 6.6075663566589355 en época 3 en el punto (-0.281517893075943, 0.24716606736183167)
Pérdida 4.461790084838867 en época 4 en el punto (-0.2978631556034088, 0.2135833203792572)
Pérdida 3.243472099304199 en época 5 en el punto (-0.31014400720596313, 0.18861114978790283)
Pérdida 2.5706567764282227 en época 6 en el punto (-0.3189893662929535, 0.1701267808675766)
Pérdida 1.825527548789978 en época 11 en el punto (-0.33284640312194824, 0.12844742834568024)
Pérdida 1.7755670547485352 en época 16 en el punto (-0.32793840765953064, 0.1157766580581665)
Pérdida 1.7500137090682983 en época 21 en el punto (-0.3194332718849182, 0.10847838222980499)
Pérdida 1.7252311706542969 en época 26 en el punto (-0.3102116286754608, 0.10224694013595581)
Pérdida 1.70042

## Descenso de gradiente en __Three-Hump Camel__

In [162]:
optimo, historial = gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), three_hump_camel, 1e-3)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 1.2378718852996826 en época 1 en el punto (-1.062302827835083, 1.1302348375320435)
Pérdida 1.236114501953125 en época 2 en el punto (-1.062865972518921, 1.1290366649627686)
Pérdida 1.234363079071045 en época 3 en el punto (-1.0634300708770752, 1.1278414726257324)
Pérdida 1.2326180934906006 en época 4 en el punto (-1.063995122909546, 1.126649260520935)
Pérdida 1.2308788299560547 en época 5 en el punto (-1.0645612478256226, 1.125459909439087)
Pérdida 1.2291451692581177 en época 6 en el punto (-1.0651283264160156, 1.1242735385894775)
Pérdida 1.2205653190612793 en época 11 en el punto (-1.067978858947754, 1.1183857917785645)
Pérdida 1.212126612663269 en época 16 en el punto (-1.0708553791046143, 1.112570881843567)
Pérdida 1.2038252353668213 en época 21 en el punto (-1.073758840560913, 1.1068283319473267)
Pérdida 1.195655107498169 en época 26 en el punto (-1.0766903162002563, 1.101157546043396)
Pérdida 1.1876122951507568 en é

In [163]:
optimo, historial = gd(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), three_hump_camel, 1e-3)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 1.2260770797729492 en época 1 en el punto (-1.4191514253616333, -0.06269367039203644)
Pérdida 1.2234282493591309 en época 2 en el punto (-1.4196600914001465, -0.06114913150668144)
Pérdida 1.2207852602005005 en época 3 en el punto (-1.4201709032058716, -0.05960717424750328)
Pérdida 1.2181482315063477 en época 4 en el punto (-1.420683741569519, -0.05806778743863106)
Pérdida 1.2155168056488037 en época 5 en el punto (-1.4211987257003784, -0.056530967354774475)
Pérdida 1.2128918170928955 en época 6 en el punto (-1.4217157363891602, -0.054996706545352936)
Pérdida 1.1998488903045654 en época 11 en el punto (-1.424331784248352, -0.04736353084445)
Pérdida 1.1869438886642456 en época 16 en el punto (-1.4269990921020508, -0.03979325294494629)
Pérdida 1.174171805381775 en época 21 en el punto (-1.4297164678573608, -0.03228498995304108)
Pérdida 1.1615300178527832 en época 26 en el punto (-1.432483196258545, -0.024837881326675415)
P

## Descenso de gradiente en __Booth__

In [164]:
optimo, historial = gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), booth, 1e-3)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 69.53148651123047 en época 1 en el punto (-1.0261746644973755, 1.166615605354309)
Pérdida 67.05146789550781 en época 2 en el punto (-0.9912458658218384, 1.201158881187439)
Pérdida 64.65992736816406 en época 3 en el punto (-0.9569426774978638, 1.2350772619247437)
Pérdida 62.35370635986328 en época 4 en el punto (-0.9232538938522339, 1.2683820724487305)
Pérdida 60.1297607421875 en época 5 en el punto (-0.8901684284210205, 1.301084280014038)
Pérdida 57.98515701293945 en época 6 en el punto (-0.8576754331588745, 1.3331947326660156)
Pérdida 48.356597900390625 en época 11 en el punto (-0.703731119632721, 1.4852380752563477)
Pérdida 40.3272705078125 en época 16 en el punto (-0.5630788207054138, 1.6240081787109375)
Pérdida 33.631553649902344 en época 21 en el punto (-0.4345652759075165, 1.7506582736968994)
Pérdida 28.047931671142578 en época 26 en el punto (-0.3171374201774597, 1.8662413358688354)
Pérdida 23.391685485839844 en é

In [165]:
optimo, historial = gd(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), booth, 1e-3)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 135.487548828125 en época 1 en el punto (-1.3699445724487305, -0.014249227941036224)
Pérdida 130.6605224609375 en época 2 en el punto (-1.3221311569213867, 0.03485282510519028)
Pérdida 126.00563049316406 en época 3 en el punto (-1.275188684463501, 0.08308134973049164)
Pérdida 121.51681518554688 en época 4 en el punto (-1.2291014194488525, 0.13045205175876617)
Pérdida 117.18809509277344 en época 5 en el punto (-1.1838539838790894, 0.17698034644126892)
Pérdida 113.0137939453125 en época 6 en el punto (-1.1394312381744385, 0.22268137335777283)
Pérdida 94.27227783203125 en época 11 en el punto (-0.9291772246360779, 0.43928879499435425)
Pérdida 78.64309692382812 en época 16 en el punto (-0.7374212145805359, 0.6373350024223328)
Pérdida 65.60932922363281 en época 21 en el punto (-0.562554657459259, 0.8184289336204529)
Pérdida 54.73986053466797 en época 26 en el punto (-0.40310898423194885, 0.9840400218963623)
Pérdida 45.675251

## Descenso de gradiente en __Styblinski-Tang__

In [166]:
optimo, historial = gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), styblinski_tang, 1e-3)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida -17.630502700805664 en época 1 en el punto (-1.0788346529006958, 1.1441421508789062)
Pérdida -18.08616065979004 en época 2 en el punto (-1.096084713935852, 1.156952977180481)
Pérdida -18.549835205078125 en época 3 en el punto (-1.1134884357452393, 1.169866919517517)
Pérdida -19.021472930908203 en época 4 en el punto (-1.1310430765151978, 1.1828826665878296)
Pérdida -19.501014709472656 en época 5 en el punto (-1.1487460136413574, 1.1959985494613647)
Pérdida -19.988372802734375 en época 6 en el punto (-1.16659414768219, 1.2092130184173584)
Pérdida -22.538902282714844 en época 11 en el punto (-1.2578948736190796, 1.2766979932785034)
Pérdida -25.265640258789062 en época 16 en el punto (-1.3522623777389526, 1.3463408946990967)
Pérdida -28.143043518066406 en época 21 en el punto (-1.4490931034088135, 1.4178234338760376)
Pérdida -31.136293411254883 en época 26 en el punto (-1.5476576089859009, 1.490765929222107)
Pérdida -34.201

In [167]:
optimo, historial = gd(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), styblinski_tang, 1e-3)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida -17.81546401977539 en época 1 en el punto (-1.4381330013275146, -0.06776811927556992)
Pérdida -18.208515167236328 en época 2 en el punto (-1.457694411277771, -0.0713517889380455)
Pérdida -18.604772567749023 en época 3 en el punto (-1.4773226976394653, -0.07499269396066666)
Pérdida -19.004005432128906 en época 4 en el punto (-1.497011423110962, -0.07869173586368561)
Pérdida -19.405986785888672 en época 5 en el punto (-1.516753911972046, -0.08244983106851578)
Pérdida -19.810461044311523 en época 6 en el punto (-1.5365432500839233, -0.08626791089773178)
Pérdida -21.861173629760742 en época 11 en el punto (-1.635943055152893, -0.10629153251647949)
Pérdida -23.93352508544922 en época 16 en el punto (-1.7354059219360352, -0.12796209752559662)
Pérdida -25.99082374572754 en época 21 en el punto (-1.8339418172836304, -0.15141160786151886)
Pérdida -27.99653434753418 en época 26 en el punto (-1.9305379390716553, -0.176781311631202

# Descenso de gradiente con momento

In [168]:
# este es casi lo mismo, solo necesita unos ajustes
def gd_con_momento(punto_partida, criterio, lr=1e-03, momento=0.9, n_epocas=100):
    print(f"Punto de partida: {punto_partida[0]}, {punto_partida[1]}")
    punto_actual = punto_partida.clone().detach().requires_grad_(True)
    cambio = 0
    historial = [punto_actual.detach()]
    for epoca in range(n_epocas):

        if punto_actual.grad is not None:
            punto_actual.grad.zero_()

        perdida = criterio(*punto_actual)
        perdida.backward()
        with torch.no_grad():

            nuevo_cambio = lr*punto_actual.grad + momento*cambio
            punto_actual -= nuevo_cambio
            cambio = nuevo_cambio
        

        historial.append(punto_actual.clone().detach())

        if epoca % 5 == 0 or epoca < 5:
            print(f"Pérdida {perdida.item()} en época {epoca + 1} en el punto ({punto_actual[0]}, {punto_actual[1]})")

    return punto_actual, historial

## Descenso de gradiente con momento en __Rosenbrock__

In [169]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), rosenbrock, 0.9, 1e-5)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 4.252490520477295 en época 1 en el punto (1.0659189224243164, 0.38573896884918213)
Pérdida 56.320987701416016 en época 2 en el punto (-287.021240234375, 135.4656982421875)
Pérdida 676436049920.0 en época 3 en el punto (8498257408.0, 14804365.0)
Pérdida inf en época 4 en el punto (-2.2094904695107755e+32, 1.29996673460027e+22)
Pérdida inf en época 5 en el punto (inf, inf)
Pérdida nan en época 6 en el punto (nan, nan)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en época 61 en el punto (nan, na

No parece converger, si aumentamos considerablemente la fricción encontramos que aun así no converge, el momento explota y no hay forma de contrarrestarlo

In [170]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), rosenbrock, 0.15, 1e-5)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 4.252490520477295 en época 1 en el punto (-0.7071306705474854, 1.0071531534194946)
Pérdida 28.63129997253418 en época 2 en el punto (-21.710968017578125, -14.206429481506348)
Pérdida 23578584.0 en época 3 en el punto (632520.1875, 14552.970703125)
Pérdida 1.6006541898191985e+25 en época 4 en el punto (-1.5183589275540128e+19, 12002453880832.0)
Pérdida inf en época 5 en el punto (inf, inf)
Pérdida nan en época 6 en el punto (nan, nan)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en época 61 en

In [171]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), rosenbrock, 0.9, 1e-5)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 437.1572570800781 en época 1 en el punto (1063.578857421875, 373.7586975097656)
Pérdida 127876804378624.0 en época 2 en el punto (-432980230144.0, 203549088.0)
Pérdida inf en época 3 en el punto (2.9221782759714326e+37, 3.3744937917000117e+25)
Pérdida inf en época 4 en el punto (-inf, inf)
Pérdida nan en época 5 en el punto (nan, nan)
Pérdida nan en época 6 en el punto (nan, nan)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en época 61 en el punto (nan, nan)
Pérdida nan en época 66 en el pun

In [172]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), rosenbrock, 0.05, 1e-5)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 437.1572570800781 en época 1 en el punto (57.74789047241211, 20.70370101928711)
Pérdida 1098339200.0 en época 2 en el punto (-3827611.0, 33161.85546875)
Pérdida 2.1464026941258627e+28 en época 3 en el punto (1.1215364113849112e+21, 146506065641472.0)
Pérdida inf en época 4 en el punto (-inf, inf)
Pérdida nan en época 5 en el punto (nan, nan)
Pérdida nan en época 6 en el punto (nan, nan)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en época 61 en el punto (nan, nan)
Pérdida nan en época 66 en

## Descenso de gradiente con momento en __Three-Hump Camel__

In [173]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), three_hump_camel, 0.9, 1e-3)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 1.2378718852996826 en época 1 en el punto (-1.5676980018615723, 0.05041778087615967)
Pérdida 0.970805287361145 en época 2 en el punto (-2.0115554332733154, 1.369512915611267)
Pérdida 1.0636075735092163 en época 3 en el punto (2.871513605117798, 0.7161086797714233)
Pérdida 41.107025146484375 en época 4 en el punto (-94.31543731689453, -3.157902717590332)
Pérdida 117229748224.0 en época 5 en el punto (6713536512.0, 87.40633392333984)
Pérdida nan en época 6 en el punto (-inf, -6042182656.0)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en épo

In [174]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), three_hump_camel, 0.1, 1e-3)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 1.2378718852996826 en época 1 en el punto (-1.1179580688476562, 1.0113228559494019)
Pérdida 1.0770268440246582 en época 2 en el punto (-1.184178113937378, 0.9207339882850647)
Pérdida 0.9568606019020081 en época 3 en el punto (-1.267221212387085, 0.8549144268035889)
Pérdida 0.8417033553123474 en época 4 en el punto (-1.3738081455230713, 0.8105878233909607)
Pérdida 0.6984512805938721 en época 5 en el punto (-1.50508713722229, 0.7858067750930786)
Pérdida 0.5146538615226746 en época 6 en el punto (-1.6413958072662354, 0.7791293263435364)
Pérdida 0.3007076382637024 en época 11 en el punto (-1.74382746219635, 0.8361953496932983)
Pérdida 0.29888951778411865 en época 16 en el punto (-1.7462711334228516, 0.8607037663459778)
Pérdida 0.29866886138916016 en época 21 en el punto (-1.7471081018447876, 0.8692306876182556)
Pérdida 0.2986433506011963 en época 26 en el punto (-1.7473981380462646, 0.872195839881897)
Pérdida 0.2986392378807

In [175]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), three_hump_camel, 0.9, 1e-3)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 1.2260770797729492 en época 1 en el punto (-1.8745445013046265, 1.328173041343689)
Pérdida 0.5685794353485107 en época 2 en el punto (-0.389293909072876, 0.6259440779685974)
Pérdida 0.4276936650276184 en época 3 en el punto (0.2353365421295166, -0.15109294652938843)
Pérdida 0.09484560042619705 en época 4 en el punto (-0.42664891481399536, -0.0917055681347847)
Pérdida 0.3778083920478821 en época 5 en el punto (0.9103187918663025, 0.4574078321456909)
Pérdida 1.6567668914794922 en época 6 en el punto (-0.48827940225601196, -1.1846641302108765)
Pérdida nan en época 11 en el punto (inf, 15539033088.0)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)


In [176]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), three_hump_camel, 0.1, 1e-3)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 1.2260770797729492 en época 1 en el punto (-1.469300389289856, 0.09047186374664307)
Pérdida 0.9762260317802429 en época 2 en el punto (-1.538130760192871, 0.21946224570274353)
Pérdida 0.7722351551055908 en época 3 en el punto (-1.6123384237289429, 0.3295118510723114)
Pérdida 0.6086559891700745 en época 4 en el punto (-1.671219825744629, 0.4249533712863922)
Pérdida 0.496798574924469 en época 5 en el punto (-1.702039361000061, 0.5071800947189331)
Pérdida 0.4279767870903015 en época 6 en el punto (-1.7144734859466553, 0.5760302543640137)
Pérdida 0.31447863578796387 en época 11 en el punto (-1.737004280090332, 0.769723117351532)
Pérdida 0.3005616068840027 en época 16 en el punto (-1.7439723014831543, 0.8375450372695923)
Pérdida 0.2988719940185547 en época 21 en el punto (-1.7463173866271973, 0.8611737489700317)
Pérdida 0.29866766929626465 en época 26 en el punto (-1.7471239566802979, 0.8693941831588745)
Pérdida 0.2986430525

A este no le fue __tan__ mal, aunque terminó alejandose del óptimo

## Descenso de gradiente con momento en __Booth__

In [177]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), booth, 0.9, 1e-5)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 69.53148651123047 en época 1 en el punto (30.947586059570312, 32.79304504394531)
Pérdida 16060.255859375 en época 2 en el punto (-453.09027099609375, -450.96661376953125)
Pérdida 3710553.0 en época 3 en el punto (6902.27685546875, 6904.177734375)
Pérdida 857284864.0 en época 4 en el punto (-104897.625, -104895.5390625)
Pérdida 198066798592.0 en época 5 en el punto (1594458.25, 1594460.125)
Pérdida 45761286897664.0 en época 6 en el punto (-24235730.0, -24235728.0)
Pérdida 3.0125264760467617e+25 en época 11 en el punto (19664025419776.0, 19664025419776.0)
Pérdida 1.9831861227318553e+37 en época 16 en el punto (-1.595470426789013e+19, -1.595470426789013e+19)
Pérdida inf en época 21 en el punto (1.2945087969917019e+25, 1.2945087969917019e+25)
Pérdida inf en época 26 en el punto (-1.0503193459993044e+31, -1.0503193459993044e+31)
Pérdida inf en época 31 en el punto (8.521925038377398e+36, 8.521925038377398e+36)
Pérdida nan en 

Pérdida nan en época 66 en el punto (nan, nan)
Pérdida nan en época 71 en el punto (nan, nan)
Pérdida nan en época 76 en el punto (nan, nan)
Pérdida nan en época 81 en el punto (nan, nan)
Pérdida nan en época 86 en el punto (nan, nan)
Pérdida nan en época 91 en el punto (nan, nan)
Pérdida nan en época 96 en el punto (nan, nan)


In [178]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), booth, 0.15, 1e-5)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 69.53148651123047 en época 1 en el punto (4.2731475830078125, 6.4083709716796875)
Pérdida 200.9012451171875 en época 2 en el punto (-4.726566314697266, -2.6319103240966797)
Pérdida 580.5718994140625 en época 3 en el punto (10.62148666381836, 12.687745094299316)
Pérdida 1677.8111572265625 en época 4 en el punto (-15.43588638305664, -13.389506340026855)
Pérdida 4848.7802734375 en época 5 en el punto (28.885093688964844, 30.917556762695312)
Pérdida 14012.7119140625 en época 6 en el punto (-46.44317626953125, -44.420448303222656)
Pérdida 2824692.75 en época 11 en el punto (674.4307861328125, 676.4346313476562)
Pérdida 569403456.0 en época 16 en el punto (-9560.33984375, -9558.3388671875)
Pérdida 114780758016.0 en época 21 en el punto (135752.046875, 135754.046875)
Pérdida 23137573928960.0 en época 26 en el punto (-1927380.0, -1927378.0)
Pérdida 4664087522836480.0 en época 31 en el punto (27364782.0, 27364784.0)
Pérdida 9.401

In [179]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), booth, 0.9, 1e-5)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 135.487548828125 en época 1 en el punto (42.41169357299805, 44.92817306518555)
Pérdida 31255.03125 en época 2 en el punto (-632.1759643554688, -630.589111328125)
Pérdida 7221122.0 en época 3 en el punto (9628.2421875, 9630.5732421875)
Pérdida 1668365696.0 en época 4 en el punto (-146335.34375, -146333.609375)
Pérdida 385458569216.0 en época 5 en el punto (2224314.0, 2224316.25)
Pérdida 89056230768640.0 en época 6 en el punto (-33809536.0, -33809532.0)
Pérdida 5.862690705948556e+25 en época 11 en el punto (27431876427776.0, 27431876427776.0)
Pérdida 3.859488510394627e+37 en época 16 en el punto (-2.2257270944557957e+19, -2.2257270944557957e+19)
Pérdida inf en época 21 en el punto (1.8058772152351292e+25, 1.8058772152351292e+25)
Pérdida inf en época 26 en el punto (-1.465225830498176e+31, -1.465225830498176e+31)
Pérdida inf en época 31 en el punto (1.188833029743379e+37, 1.188833029743379e+37)
Pérdida nan en época 36 en e

In [180]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), booth, 0.15, 1e-5)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 135.487548828125 en época 1 en el punto (5.886411666870117, 7.434494972229004)
Pérdida 391.0589599609375 en época 2 en el punto (-6.7645263671875, -5.080866813659668)
Pérdida 1129.894287109375 en época 3 en el punto (14.579179763793945, 16.357738494873047)
Pérdida 3265.21337890625 en época 4 en el punto (-21.818662643432617, -19.973670959472656)
Pérdida 9436.232421875 en época 5 en el punto (39.97737121582031, 41.86886978149414)
Pérdida 27270.171875 en época 6 en el punto (-65.1307144165039, -63.206668853759766)
Pérdida 5497137.0 en época 11 en el punto (940.4632568359375, 942.4506225585938)
Pérdida 1108116736.0 en época 16 en el punto (-13337.326171875, -13335.328125)
Pérdida 223374934016.0 en época 21 en el punto (189377.390625, 189379.390625)
Pérdida 45028055449600.0 en época 26 en el punto (-2688747.75, -2688745.75)
Pérdida 9076785241128960.0 en época 31 en el punto (38174616.0, 38174620.0)
Pérdida 1.829704996363436

## Descenso de gradiente con momento en __Styblinski-Tang__

In [181]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), styblinski_tang, 0.9, 1e-5)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida -17.630502700805664 en época 1 en el punto (-16.44639778137207, 12.566986083984375)
Pérdida 45614.59765625 en época 2 en el punto (7751.75341796875, -3381.166259765625)
Pérdida 1870732866879488.0 en época 3 en el punto (-838440583168.0, 69577965568.0)
Pérdida inf en época 4 en el punto (1.0609364806404248e+36, -6.06300167568119e+32)
Pérdida nan en época 5 en el punto (-inf, inf)
Pérdida nan en época 6 en el punto (nan, nan)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en época 61 en el punto 

In [182]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), styblinski_tang, 0.1, 1e-5)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida -17.630502700805664 en época 1 en el punto (-2.7711472511291504, 2.402052640914917)
Pérdida -62.384525299072266 en época 2 en el punto (-3.19892954826355, 3.22344970703125)
Pérdida -58.587669372558594 en época 3 en el punto (-2.020195960998535, 1.4322439432144165)
Pérdida -40.09796905517578 en época 4 en el punto (-3.853536605834961, 2.8862175941467285)
Pérdida -42.904266357421875 en época 5 en el punto (1.175593376159668, 2.44559645652771)
Pérdida -31.009899139404297 en época 6 en el punto (2.481654405593872, 3.183152198791504)
Pérdida -31.111919403076172 en época 11 en el punto (3.0353996753692627, 0.9784951210021973)
Pérdida -37.12638473510742 en época 16 en el punto (3.3430795669555664, 3.341648578643799)
Pérdida -7.96726131439209 en época 21 en el punto (1.9589546918869019, 1.9622650146484375)
Pérdida -36.52146911621094 en época 26 en el punto (0.9128069877624512, 0.9102826118469238)
Pérdida -40.434505462646484 en é

In [183]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), styblinski_tang, 0.9, 1e-5)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida -17.81546401977539 en época 1 en el punto (-18.95795440673828, -3.238830804824829)
Pérdida 61626.0078125 en época 2 en el punto (11970.2138671875, 9.027719497680664)
Pérdida 1.0265441062617088e+16 en época 3 en el punto (-3087295315968.0, -1187.5848388671875)
Pérdida inf en época 4 en el punto (5.296719830258748e+37, 3014836480.0)
Pérdida nan en época 5 en el punto (nan, -4.932462206509274e+28)
Pérdida nan en época 6 en el punto (nan, inf)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en époc

In [184]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), styblinski_tang, 0.15, 1e-5)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida -17.81546401977539 en época 1 en el punto (-4.34186315536499, -0.593339204788208)
Pérdida 11.788061141967773 en época 2 en el punto (9.418184280395508, -2.329692840576172)
Pérdida 3213.452392578125 en época 3 en el punto (-218.97714233398438, -4.5026726722717285)
Pérdida 1149264512.0 en época 4 en el punto (3149306.25, 11.702131271362305)
Pérdida 4.9184650569138164e+25 en época 5 en el punto (-9.370569156023288e+18, -441.3341369628906)
Pérdida nan en época 6 en el punto (inf, 25786868.0)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)


Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en época 61 en el punto (nan, nan)
Pérdida nan en época 66 en el punto (nan, nan)
Pérdida nan en época 71 en el punto (nan, nan)
Pérdida nan en época 76 en el punto (nan, nan)
Pérdida nan en época 81 en el punto (nan, nan)
Pérdida nan en época 86 en el punto (nan, nan)
Pérdida nan en época 91 en el punto (nan, nan)
Pérdida nan en época 96 en el punto (nan, nan)


# Método __ADAM__

In [193]:
# este es casi lo mismo, solo necesita unos ajustes
def adam(punto_partida, criterio, lr=1e-03, beta1=0.9, beta2=0.999, eps=1e-8, n_epocas=100):
    print(f"Punto de partida: {punto_partida[0]}, {punto_partida[1]}")
    punto_actual = punto_partida.clone().detach().requires_grad_(True)
    m = torch.zeros_like(punto_actual) # primer momento
    nu = torch.zeros_like(punto_actual) # segundo momento
    historial = [punto_actual.detach()]
    for epoca in range(n_epocas):

        if punto_actual.grad is not None:
            punto_actual.grad.zero_()

        perdida = criterio(*punto_actual)
        perdida.backward()
        with torch.no_grad():

            g = punto_actual.grad.clone()
            m = beta1*m + (1 - beta1)*g
            nu = beta2*nu + (1 - beta2)*g**2

            mhat = m / (1 - beta1**(epoca + 1)) # Evitar division por 0
            nuhat = nu / (1 - beta2**(epoca + 1))
            
            punto_actual -= lr*mhat / (nuhat.sqrt() + eps)        

        historial.append(punto_actual.clone().detach())

        if epoca % 5 == 0 or epoca < 5:
            print(f"Pérdida {perdida.item()} en época {epoca + 1} en el punto ({punto_actual[0]}, {punto_actual[1]})")

    return punto_actual, historial

## ADAM para __Rosenbrock__

In [194]:
optimo, historial = adam(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), rosenbrock)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 4.252490520477295 en época 1 en el punto (-1.06074059009552, 1.1304359436035156)
Pérdida 4.249423503875732 en época 2 en el punto (-1.0597525835037231, 1.1294368505477905)
Pérdida 4.246627330780029 en época 3 en el punto (-1.0587894916534424, 1.1284369230270386)
Pérdida 4.2440924644470215 en época 4 en el punto (-1.0578685998916626, 1.1274343729019165)
Pérdida 4.24179220199585 en época 5 en el punto (-1.0570107698440552, 1.1264275312423706)
Pérdida 4.239675521850586 en época 6 en el punto (-1.0562373399734497, 1.1254150867462158)
Pérdida 4.229671955108643 en época 11 en el punto (-1.0538263320922852, 1.1202625036239624)
Pérdida 4.219803810119629 en época 16 en el punto (-1.0521823167800903, 1.115099310874939)
Pérdida 4.20991849899292 en época 21 en el punto (-1.0495176315307617, 1.1100811958312988)
Pérdida 4.199943542480469 en época 26 en el punto (-1.046680212020874, 1.105055570602417)
Pérdida 4.1899614334106445 en époc

In [195]:
optimo, historial = adam(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), rosenbrock)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 437.1572570800781 en época 1 en el punto (-1.4176448583602905, -0.06324079632759094)
Pérdida 435.5603942871094 en época 2 en el punto (-1.4166449308395386, -0.0622408464550972)
Pérdida 433.96746826171875 en época 3 en el punto (-1.4156451225280762, -0.06124097481369972)
Pérdida 432.37847900390625 en época 4 en el punto (-1.4146454334259033, -0.0602412186563015)
Pérdida 430.79339599609375 en época 5 en el punto (-1.4136459827423096, -0.059241607785224915)
Pérdida 429.2124938964844 en época 6 en el punto (-1.412646770477295, -0.05824217200279236)
Pérdida 421.3704833984375 en época 11 en el punto (-1.4076558351516724, -0.05324874073266983)
Pérdida 413.6377258300781 en época 16 en el punto (-1.4026771783828735, -0.048264242708683014)
Pérdida 406.0196228027344 en época 21 en el punto (-1.397715449333191, -0.043292008340358734)
Pérdida 398.52069091796875 en época 26 en el punto (-1.3927745819091797, -0.03833485394716263)
Pérd

Aquí le fue mucho peor que al descenso gradiente normal, aunque no iba mal, con el fin de una comparación justa no aumentaremos las épocas

## ADAM para __Three-Hump Camel__

In [196]:
optimo, historial = adam(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), three_hump_camel)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 1.2378718852996826 en época 1 en el punto (-1.062740683555603, 1.1304359436035156)
Pérdida 1.2361085414886475 en época 2 en el punto (-1.063740849494934, 1.1294360160827637)
Pérdida 1.2343453168869019 en época 3 en el punto (-1.0647411346435547, 1.1284362077713013)
Pérdida 1.2325823307037354 en época 4 en el punto (-1.065741777420044, 1.1274365186691284)
Pérdida 1.230819582939148 en época 5 en el punto (-1.0667427778244019, 1.1264370679855347)
Pérdida 1.2290570735931396 en época 6 en el punto (-1.067744255065918, 1.12543785572052)
Pérdida 1.2202459573745728 en época 11 en el punto (-1.0727614164352417, 1.120446801185608)
Pérdida 1.211437463760376 en época 16 en el punto (-1.07780122756958, 1.1154680252075195)
Pérdida 1.2026307582855225 en época 21 en el punto (-1.0828721523284912, 1.110506296157837)
Pérdida 1.193823218345642 en época 26 en el punto (-1.0879803895950317, 1.1055654287338257)
Pérdida 1.185012698173523 en ép

In [197]:
optimo, historial = adam(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), three_hump_camel)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 1.2260770797729492 en época 1 en el punto (-1.4196449518203735, -0.06324079632759094)
Pérdida 1.2240225076675415 en época 2 en el punto (-1.420644998550415, -0.06224081292748451)
Pérdida 1.221967339515686 en época 3 en el punto (-1.4216452836990356, -0.06124085932970047)
Pérdida 1.219910979270935 en época 4 en el punto (-1.4226458072662354, -0.06024094298481941)
Pérdida 1.217853307723999 en época 5 en el punto (-1.4236466884613037, -0.05924107879400253)
Pérdida 1.2157950401306152 en época 6 en el punto (-1.4246479272842407, -0.05824127420783043)
Pérdida 1.2054849863052368 en época 11 en el punto (-1.4296613931655884, -0.05324355140328407)
Pérdida 1.1951487064361572 en época 16 en el punto (-1.4346917867660522, -0.04824891686439514)
Pérdida 1.1847877502441406 en época 21 en el punto (-1.439743995666504, -0.043258488178253174)
Pérdida 1.1744027137756348 en época 26 en el punto (-1.4448211193084717, -0.03827320784330368)
P

## ADAM para __Booth__

In [198]:
optimo, historial = adam(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), booth)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida 69.53148651123047 en época 1 en el punto (-1.06074059009552, 1.1324360370635986)
Pérdida 69.46075439453125 en época 2 en el punto (-1.059740662574768, 1.1334359645843506)
Pérdida 69.39006805419922 en época 3 en el punto (-1.0587407350540161, 1.1344358921051025)
Pérdida 69.31941223144531 en época 4 en el punto (-1.0577408075332642, 1.1354358196258545)
Pérdida 69.2488021850586 en época 5 en el punto (-1.0567408800125122, 1.1364357471466064)
Pérdida 69.17822265625 en época 6 en el punto (-1.0557410717010498, 1.1374355554580688)
Pérdida 68.8259506225586 en época 11 en el punto (-1.0507428646087646, 1.142433762550354)
Pérdida 68.47472381591797 en época 16 en el punto (-1.045746922492981, 1.1474294662475586)
Pérdida 68.1246337890625 en época 21 en el punto (-1.0407545566558838, 1.1524218320846558)
Pérdida 67.77571105957031 en época 26 en el punto (-1.0357661247253418, 1.1574100255966187)
Pérdida 67.42799377441406 en época 31 e

In [199]:
optimo, historial = adam(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), booth)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida 135.487548828125 en época 1 en el punto (-1.4176448583602905, -0.06324079632759094)
Pérdida 135.38888549804688 en época 2 en el punto (-1.416644811630249, -0.062240805476903915)
Pérdida 135.2902374267578 en época 3 en el punto (-1.415644884109497, -0.06124082952737808)
Pérdida 135.191650390625 en época 4 en el punto (-1.4146449565887451, -0.06024087592959404)
Pérdida 135.0930938720703 en época 5 en el punto (-1.4136450290679932, -0.059240952134132385)
Pérdida 134.99456787109375 en época 6 en el punto (-1.4126451015472412, -0.05824106186628342)
Pérdida 134.5025634765625 en época 11 en el punto (-1.4076465368270874, -0.053242333233356476)
Pérdida 134.01161193847656 en época 16 en el punto (-1.4026495218276978, -0.04824531450867653)
Pérdida 133.52175903320312 en época 21 en el punto (-1.3976550102233887, -0.04325065016746521)
Pérdida 133.03311157226562 en época 26 en el punto (-1.3926634788513184, -0.0382588729262352)
Pérd

## ADAM para __Styblinski-Tang__

In [200]:
optimo, historial = adam(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), styblinski_tang)

Punto de partida: -1.0617406368255615, 1.1314359903335571
Pérdida -17.630502700805664 en época 1 en el punto (-1.062740683555603, 1.1324360370635986)
Pérdida -17.66031265258789 en época 2 en el punto (-1.0637407302856445, 1.1334360837936401)
Pérdida -17.690139770507812 en época 3 en el punto (-1.064740777015686, 1.1344361305236816)
Pérdida -17.719985961914062 en época 4 en el punto (-1.0657408237457275, 1.1354361772537231)
Pérdida -17.749847412109375 en época 5 en el punto (-1.0667409896850586, 1.1364363431930542)
Pérdida -17.779733657836914 en época 6 en el punto (-1.0677411556243896, 1.1374365091323853)
Pérdida -17.92943572998047 en época 11 en el punto (-1.0727429389953613, 1.1424387693405151)
Pérdida -18.079648971557617 en época 16 en el punto (-1.0777473449707031, 1.1474440097808838)
Pérdida -18.230396270751953 en época 21 en el punto (-1.0827550888061523, 1.1524534225463867)
Pérdida -18.381704330444336 en época 26 en el punto (-1.0877671241760254, 1.1574678421020508)
Pérdida -18.

In [201]:
optimo, historial = adam(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), styblinski_tang)

Punto de partida: -1.418644905090332, -0.06424079835414886
Pérdida -17.81546401977539 en época 1 en el punto (-1.4196449518203735, -0.06524080038070679)
Pérdida -17.838489532470703 en época 2 en el punto (-1.420644998550415, -0.06624091416597366)
Pérdida -17.861534118652344 en época 3 en el punto (-1.4216450452804565, -0.06724122166633606)
Pérdida -17.884601593017578 en época 4 en el punto (-1.422645092010498, -0.06824179738759995)
Pérdida -17.907690048217773 en época 5 en el punto (-1.4236451387405396, -0.06924270838499069)
Pérdida -17.930801391601562 en época 6 en el punto (-1.424645185470581, -0.07024403661489487)
Pérdida -18.046676635742188 en época 11 en el punto (-1.4296458959579468, -0.07525929808616638)
Pérdida -18.16313362121582 en época 16 en el punto (-1.434647560119629, -0.08029485493898392)
Pérdida -18.28021240234375 en época 21 en el punto (-1.439650297164917, -0.08535778522491455)
Pérdida -18.39794158935547 en época 26 en el punto (-1.4446545839309692, -0.090453810989856

En general parece que este algoritmo tarda más en converger que el descenso gradiente, al menos con estas funciones

# Visualizando los recorridos

In [202]:
def plot_trayectorias(funcion, historiales, etiquetas, rango=(-5, 5), puntos=100):
    x = np.linspace(*rango, puntos)
    y = np.linspace(*rango, puntos)
    X, Y = np.meshgrid(x, y)
    Z = funcion(torch.tensor(X, dtype=torch.float32), 
                torch.tensor(Y, dtype=torch.float32)).detach().numpy()

    fig = plt.figure(figsize=(14, 6))

    ax1 = fig.add_subplot(121, projection='3d')
    ax1.plot_surface(X, Y, Z, alpha=0.4, cmap='viridis')

    for historial, etiqueta in zip(historiales, etiquetas):
        tray = torch.stack(historial).numpy()
        xs, ys = tray[:, 0], tray[:, 1]
        zs = [funcion(torch.tensor(p, dtype=torch.float32),
                      torch.tensor(q, dtype=torch.float32)).item() 
              for p, q in zip(xs, ys)]
        ax1.plot(xs, ys, zs, marker='o', markersize=2, label=etiqueta)
        ax1.scatter(*tray[-1], zs[-1], marker='*', s=100)  # punto final

    ax1.set_title("Trayectoria 3D")
    ax1.set_xlabel("x"); ax1.set_ylabel("y"); ax1.set_zlabel("z")
    ax1.legend()

    ax2 = fig.add_subplot(122)
    ax2.contourf(X, Y, Z, levels=50, cmap='viridis')

    for historial, etiqueta in zip(historiales, etiquetas):
        tray = torch.stack(historial).numpy()
        ax2.plot(tray[:, 0], tray[:, 1], marker='o', markersize=2, label=etiqueta)
        ax2.scatter(*tray[-1], marker='*', s=100, zorder=5)  # punto final

    ax2.set_title("Contorno 2D")
    ax2.set_xlabel("x"); ax2.set_ylabel("y")
    ax2.legend()

    plt.tight_layout()
    plt.show()